# U-Net Scratch Segmentation Training

## 1. Connect with Google drive

In [ ]:
"""
    Connect Google Drive for U-Net training.

    Purpose:
        - Read the prepared scratch patch dataset from Drive.
        - Use the Drive project folder as the Colab working project.
        - Save trained U-Net checkpoints back into the same Drive project.
"""
from google.colab import drive

# Mount Drive once at /content/drive. Colab will ask for permission.
drive.mount("/content/drive")


In [ ]:
"""
    Define the Drive project root used by this notebook.

    PROJECT:
        /content/drive/MyDrive/Surface-Scratch-Detection

    Expected layout:
        PROJECT/data/scratch_v2_patches.zip
        PROJECT/src/unet/train.py
        PROJECT/checkpoints/unet/
"""
from pathlib import Path
import zipfile

PROJECT = Path(
    "/content/drive/MyDrive/Surface-Scratch-Detection"
).resolve()

print("Project:", PROJECT)
print("Project exists:", PROJECT.exists())


## 2. Unzip

In [ ]:
"""
    Extract the patch dataset zip stored on Drive.

    Input:
        PROJECT/data/scratch_v2_patches.zip

    Output:
        PROJECT/data/scratch_v2_patches/

    Dataset format expected by src.unet.train:
        scratch_v2_patches/train/images
        scratch_v2_patches/train/masks
        scratch_v2_patches/valid/images
        scratch_v2_patches/valid/masks
        scratch_v2_patches/test/images
        scratch_v2_patches/test/masks
"""
zip_path   = PROJECT / "data" / "scratch_v2_patches.zip"
output_dir = PROJECT / "data"

if not zip_path.is_file():
    raise FileNotFoundError(f"Dataset zip not found: {zip_path}")

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(output_dir)

print("Unzipped:", output_dir)
print("Dataset:", output_dir / "scratch_v2_patches")


## 3. Check Struture dataset

In [ ]:
"""
    Quick folder-structure inspection after unzip.

    This cell should show:
        train/images, train/masks
        valid/images, valid/masks
        test/images,  test/masks
"""
!find "{PROJECT}/data/scratch_v2_patches" -maxdepth 2 -type d


In [ ]:
"""
    Copy the dataset from Drive to Colab local disk.

    Why:
        Training reads many image/mask files. Local /content is faster than Drive.

    Input:
        /content/drive/MyDrive/Surface-Scratch-Detection/data/scratch_v2_patches

    Output:
        /content/scratch_v2_patches
"""
!rm -rf /content/scratch_v2_patches
!cp -r /content/drive/MyDrive/Surface-Scratch-Detection/data/scratch_v2_patches /content/scratch_v2_patches

print("Local dataset copied to: /content/scratch_v2_patches")


## 4. Remove empty file

In [ ]:
"""
    Remove zero-byte image files before training.

    Why:
        Empty images usually come from interrupted upload/copy operations.
        If kept, OpenCV/PIL dataloading may fail during training.

    Behavior:
        - Scan train/valid/test image folders.
        - Delete each zero-byte image.
        - Delete the matching mask with the same stem.
"""
from pathlib import Path

root = Path("/content/scratch_v2_patches")

for split in ["train", "valid", "test"]:
    img_dir = root / split / "images"
    mask_dir = root / split / "masks"

    if not img_dir.is_dir() or not mask_dir.is_dir():
        raise FileNotFoundError(f"Missing split folders: {img_dir} / {mask_dir}")

    bad_images = [p for p in img_dir.glob("*") if p.stat().st_size == 0]
    print(split, "bad images:", len(bad_images))

    for img_path in bad_images:
        stem = img_path.stem
        print("Delete image:", img_path)
        img_path.unlink(missing_ok=True)

        for mask_path in mask_dir.glob(stem + ".*"):
            print("Delete mask:", mask_path)
            mask_path.unlink(missing_ok=True)


## 5. Train model

In [ ]:
"""
    Set Colab working directory to the project root before training.

    Why:
        The training command uses package import:
            python -m src.unet.train

        This requires PROJECT to be the current working directory and in sys.path.
"""
from pathlib import Path
import os, sys

PROJECT = Path("/content/drive/MyDrive/Surface-Scratch-Detection").resolve()

os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))

print("CWD:", Path.cwd())
print("src exists:", (PROJECT / "src").exists())
print("train.py exists:", (PROJECT / "src" / "unet" / "train.py").exists())


In [ ]:
"""
    Train U-Net for binary scratch segmentation.

    Data:
        /content/scratch_v2_patches

    Main settings:
        epochs       : 40
        batch_size   : 8
        img_size     : 512 patch input
        base_channel : 32 lighter model width for Colab T4
        norm         : group normalization, more stable for small batch size
        lr           : 1e-4

    Output:
        PROJECT/checkpoints/unet/
            best.pth
            last.pth
            results.json
            results.csv
"""
!python -m src.unet.train \
    --data /content/scratch_v2_patches \
    --epochs 40 \
    --batch_size 8 \
    --img_size 512 \
    --base_channel 32 \
    --norm group \
    --lr 0.0001 \
    --num_workers 2 \
    --save_dir checkpoints/unet


## 6. Download model

In [ ]:
"""
    Download the best U-Net checkpoint from Colab.

    Normal workflow:
        The checkpoint is already saved in Drive under:
            Surface-Scratch-Detection/checkpoints/unet/best.pth

    Use this cell when:
        You also want a direct browser download from the Colab session.
"""
from google.colab import files

model_path = (
    "/content/drive/MyDrive/Surface-Scratch-Detection/"
    "checkpoints/unet/best.pth"
)

files.download(model_path)
